# 알약 탐지·분류 통합 추론

이 노트북은 학습된 탐지기와 분류기 `best.pt`를 불러와 원본 이미지에서 알약 위치와 세부 품목을 순서대로 예측하고 결과를 저장한다. 두 모델의 학습은 개별 학습 노트북에서 수행한다.

## 실행 환경

- **Colab:** 저장소, 두 checkpoint와 추론 이미지 경로를 준비한 뒤 실행한다.
- **로컬:** 저장소 루트에서 `uv sync --locked`를 실행하고 설정 셀에 추론 이미지 폴더를 입력한다. CUDA, MPS 또는 CPU 중 사용 가능한 장치를 자동으로 선택한다.

## 추론 실행 설정

설정 셀에서 탐지기·분류기 실행 이름, 입력 경로, batch 크기와 탐지 점수 임곗값을 지정한다. 결과는 `outputs/inference/<run_name>/`에 저장된다.

주요 결과물:

- `pill_predictions.json`: bbox, 품목 예측과 결합 점수
- `pill_recognition_errors.json`: 탐지 또는 분류 단계 오류
- `submission.csv`: Kaggle 제출 형식의 최종 예측
- `visualizations/`: 이미지별 최종 예측 시각화

In [ ]:
# ruff: noqa: E402
import sys
from dataclasses import replace
from pathlib import Path

project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from configs.pill_recognition import load_pill_recognition_config
from pipeline import (
    prepare_pill_recognition_data,
    recognize_pills_from_config,
)

In [ ]:
run_name = "20261105-ssdlite320-resnet50-infer-test-full-01"
device = "mps"
show_progress = True

# tags = [
#     "task:inference",
#     "stage:full",
#     "detector:ssdlite320",
#     "classifier:resnet50",
#     "data:test-full",
#     "device:mps",
# ]

output_root = project_root / "outputs"
output_dir = output_root / "inference"

detector_run_name = "20261104-ssdlite320-det-default-baseline-01"
classifier_run_name = "resnet50_baseline_v2"
detector_backend = "torchvision"
detector_image_size = 320
detector_nms_threshold = 0.45

input_dir_value = "data/sprint_ai_project1_data/test_images"
input_dir = Path(input_dir_value).expanduser()
if not input_dir.is_absolute():
    input_dir = project_root / input_dir
batch_size = 4
score_threshold = 0.25

detector_run_dir = output_root / "detector" / detector_run_name
classifier_run_dir = output_root / "classifier" / classifier_run_name

base_config = load_pill_recognition_config(
    project_root / "configs" / "pill_recognition.yaml"
)
config = replace(
    base_config,
    run_name=run_name,
    input_dir=input_dir,
    detector_run_dir=detector_run_dir,
    classifier_run_dir=classifier_run_dir,
    output_dir=output_dir,
    detector_backend=detector_backend,
    detector_image_size=detector_image_size,
    detector_nms_threshold=detector_nms_threshold,
    batch_size=batch_size,
    score_threshold=score_threshold,
    show_progress=show_progress,
    device=device,
)
run_dir = config.run_dir
print(f"====================\nrun_name: {run_name}\n====================\n")
print(
    f"device: {config.device}\n"
    f"input_dir: {config.input_dir}\n"
    f"run_dir: {config.run_dir}\n"
    f"detector_run: {detector_run_name}\n"
    f"detector_backend: {config.detector_backend}\n"
    f"classifier_run: {classifier_run_name}"
)
print(f"batch_size: {config.batch_size}")
print(f"score_threshold: {config.score_threshold}")

## 입력 준비 및 검증

`input_dir`에 추론할 이미지를 넣으면 기본 factory가 재귀적으로 읽는다. 외부 factory를 사용할 경우 `factory(config=config, batch_size=batch_size)` 호출을 지원하고 `images: list[Tensor]`와 `meta: list[dict]`를 반환해야 한다.

In [ ]:
inference_data = prepare_pill_recognition_data(config)
device = inference_data.device
print(f"입력 검증 완료: device={device}")

## 통합 추론 실행

두 checkpoint는 현재 label map 및 class map과 일치해야 한다. `detector_backend`에 따라 TorchVision 또는 Ultralytics YOLO 체크포인트를 불러온다. 모델은 같은 CUDA, MPS 또는 CPU 장치에서 순서대로 실행되며, 탐지 bbox를 이미지 경계로 보정해 crop하고 학습과 같은 padding·정규화를 적용한다. 최종 점수는 `detector_score × classifier_score`로 계산한다.

In [ ]:
recognition_result = recognize_pills_from_config(
    config,
    data=inference_data,
)
print("통합 추론과 artifact 저장 완료")

## 결과 artifact 및 제출 파일 확인

저장된 예측과 단계별 오류 수를 확인하고 Kaggle 제출 CSV의 컬럼, 행 수와 고유 ID를 검증한다.

In [ ]:
predictions_path = run_dir / "pill_predictions.json"
errors_path = run_dir / "pill_recognition_errors.json"
submission_path = run_dir / "submission.csv"
print(
    f"predictions={len(recognition_result.predictions)}, "
    f"detector_errors={len(recognition_result.detector_errors)}, "
    f"classifier_errors={len(recognition_result.classifier_errors)}"
)
print(f"saved: {predictions_path}")
print(f"saved: {errors_path}")
print(f"saved: {submission_path}")

In [ ]:
prediction_table = pd.DataFrame(
    [prediction.to_dict() for prediction in recognition_result.predictions]
)
display(prediction_table.head(20))

submission_columns = [
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "score",
]
submission_table = pd.read_csv(submission_path)
if list(submission_table.columns) != submission_columns:
    raise ValueError("submission.csv 컬럼이 Kaggle 제출 형식과 다릅니다.")
if len(submission_table) != len(recognition_result.predictions):
    raise ValueError("제출 행 수와 최종 예측 수가 다릅니다.")
if not submission_table["annotation_id"].is_unique:
    raise ValueError("annotation_id가 중복되었습니다.")
display(submission_table.head(20))

error_table = pd.DataFrame(
    [
        {"stage": "detector", **error.to_dict()}
        for error in recognition_result.detector_errors
    ]
    + [
        {"stage": "classifier", **error.to_dict()}
        for error in recognition_result.classifier_errors
    ]
)
display(error_table.head(20))

## 검증셋 시각화 설정

검증셋 평가 후 확인할 이미지 ID를 지정한다. 빈 목록이면 FP가 포함된 이미지를 모두 표시하며, ID를 입력하면 해당 validation 이미지만 자유롭게 선택할 수 있다.

In [ ]:
validation_preview_image_ids = []
# validation_preview_image_ids = ["1", "25", "310"]
validation_preview_columns = 5

## 검증셋 통합 평가

검증셋을 동일한 탐지기·분류기에 입력해 탐지, 매칭된 crop 분류, 최종 품목 예측을 함께 평가한다. `error_table`은 실행 예외만 나타내므로 아래 성능 지표와는 구분한다.

In [ ]:
from classifier.inference import load_classifier_inference_model
from configs.detector import load_detector_config
from data_loading.detector import build_detector_inputs
from pipeline import (
    evaluate_pill_recognition,
    load_configured_detector,
    recognize_pills,
    save_pill_recognition_evaluation,
)
from utils.io import read_json
from utils.runtime import resolve_device

validation_data_dir_value = "data/detector"
validation_data_dir = Path(validation_data_dir_value).expanduser()
if not validation_data_dir.is_absolute():
    validation_data_dir = project_root / validation_data_dir
matching_iou_threshold = 0.5

validation_config = replace(
    load_detector_config(project_root / "configs" / "detector.yaml"),
    data_dir=validation_data_dir,
    batch_size=batch_size,
    device=config.device,
    show_progress=show_progress,
)
validation_device = resolve_device(validation_config.device)
_, validation_loader, dataset_label_map, validation_summary = (
    build_detector_inputs(
        config=validation_config,
        device=validation_device,
    )
)
detector_label_map = read_json(detector_run_dir / "label_map.json")
class_map = read_json(classifier_run_dir / "class_map.json")
if dataset_label_map != detector_label_map:
    raise ValueError("검증 데이터와 탐지 checkpoint의 label_map이 다릅니다.")
print(f"validation_data_dir: {validation_data_dir}")
print(f"validation batches: {len(validation_loader)}")
print(f"matching_iou_threshold: {matching_iou_threshold}")

### 검증셋 추론

학습에 사용하지 않은 validation split 전체를 추론한다. 테스트셋 추론 결과와 섞이지 않도록 결과는 현재 실행 폴더의 `validation/`에 별도로 저장한다.

In [ ]:
detector_model, validation_device, detector_predictor = load_configured_detector(
    config,
    detector_label_map,
    device=validation_device,
)
classifier_model, validation_device = load_classifier_inference_model(
    classifier_run_dir / "best.pt",
    class_map,
    device=validation_device,
)
validation_result = recognize_pills(
    detector_model,
    classifier_model,
    validation_loader,
    class_map,
    validation_device,
    score_threshold=score_threshold,
    detector_name=detector_run_name,
    show_progress=show_progress,
    detector_predictor=detector_predictor,
)
print(
    f"검증 추론 완료: predictions={len(validation_result.predictions)}, "
    f"detector_errors={len(validation_result.detector_errors)}, "
    f"classifier_errors={len(validation_result.classifier_errors)}"
)

In [ ]:
validation_evaluation, validation_details = evaluate_pill_recognition(
    validation_result,
    validation_loader,
    matching_iou_threshold=matching_iou_threshold,
)
validation_output_dir = run_dir / "validation"
metrics_path, details_path = save_pill_recognition_evaluation(
    validation_evaluation,
    validation_details,
    validation_output_dir,
)
metric_table = pd.DataFrame(
    validation_evaluation.to_dict().items(),
    columns=["metric", "value"],
)
display(metric_table)
print(f"saved: {metrics_path}")
print(f"saved: {details_path}")

### 오류 유형 분석

`correct`, `misclassified`, `false_positive`, `false_negative`별 개수를 확인한다. 오분류 표는 실제 품목과 예측 품목 조합 중 자주 틀린 순서로 정렬한다.

In [ ]:
validation_detail_table = pd.DataFrame(
    [detail.to_dict() for detail in validation_details]
)
status_table = (
    validation_detail_table["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)
display(status_table)

misclassification_table = (
    validation_detail_table.query("status == 'misclassified'")
    .groupby(
        ["target_category_id", "predicted_category_id"],
        dropna=False,
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
display(misclassification_table.head(20))
display(
    validation_detail_table.query("status != 'correct'")
    .sort_values("image_id", ascending=True, na_position="last")
    .head(40)
)

### 검증 결과 자유 시각화

validation loader의 dataset에서 지정한 이미지 ID만 다시 읽어 표시한다. ID를 지정하지 않으면 FP가 포함된 이미지를 중복 없이 모두 표시한다. 초록색 점선은 정답, 파란색은 `correct`, 빨간색은 `false_positive`, 주황색은 `misclassified`, 자홍색은 `false_negative`다. status는 `matching_iou_threshold` 기준의 진단 결과이며 mAP@0.75~0.95 판정과는 구분한다.

In [ ]:
from collections import defaultdict, deque
from math import ceil

from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle

plot_columns = validation_preview_columns
prediction_lookup = defaultdict(deque)
for prediction in validation_result.predictions:
    prediction_lookup[
        (
            prediction.image_id,
            prediction.category_id,
            prediction.detector_score,
            prediction.classifier_score,
            prediction.final_score,
        )
    ].append(prediction)
details_by_image = defaultdict(list)
prediction_details_by_image = defaultdict(list)
for detail in validation_details:
    details_by_image[detail.image_id].append(detail)
    if detail.predicted_category_id is None:
        continue
    prediction_key = (
        detail.image_id,
        detail.predicted_category_id,
        detail.detector_score,
        detail.classifier_score,
        detail.final_score,
    )
    if not prediction_lookup[prediction_key]:
        raise ValueError(f"오류 상세와 예측 bbox를 연결할 수 없습니다: {prediction_key}")
    prediction_details_by_image[detail.image_id].append(
        (detail, prediction_lookup[prediction_key].popleft())
    )

default_image_ids = list(
    dict.fromkeys(
        detail.image_id
        for detail in validation_details
        if detail.status == "false_positive"
    )
)
image_ids_to_plot = (
    [str(image_id) for image_id in validation_preview_image_ids]
    if validation_preview_image_ids
    else default_image_ids
)
if not image_ids_to_plot:
    raise ValueError("시각화할 validation 이미지가 없습니다.")

validation_dataset = validation_loader.dataset
record_index_by_image_id = {
    str(record["image_id"]): index
    for index, record in enumerate(validation_dataset.records)
}
unknown_image_ids = sorted(
    set(image_ids_to_plot) - record_index_by_image_id.keys()
)
if unknown_image_ids:
    raise ValueError(f"validation에 없는 image_id: {unknown_image_ids}")

plot_rows = ceil(len(image_ids_to_plot) / plot_columns)
figure, axes = plt.subplots(
    plot_rows,
    plot_columns,
    figsize=(4 * plot_columns, 4 * plot_rows),
    squeeze=False,
)
for axis in axes.flat:
    axis.set_axis_off()

status_colors = {
    "correct": "dodgerblue",
    "false_positive": "red",
    "misclassified": "orange",
    "false_negative": "magenta",
}
for axis, image_id in zip(axes.flat, image_ids_to_plot, strict=False):
    sample = validation_dataset[record_index_by_image_id[image_id]]
    axis.imshow(sample["image"].permute(1, 2, 0).numpy())
    for x1, y1, x2, y2 in sample["target"]["boxes"].tolist():
        axis.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                edgecolor="lime",
                linewidth=1.5,
                linestyle="--",
            )
        )
    status_counts = defaultdict(int)
    for detail in details_by_image[image_id]:
        status_counts[detail.status] += 1
        target_box = getattr(detail, "target_bbox_xyxy", None)
        if detail.status == "false_negative" and target_box is not None:
            x1, y1, x2, y2 = target_box
            axis.add_patch(
                Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    edgecolor=status_colors[detail.status],
                    linewidth=2.5,
                    linestyle=":",
                )
            )
            axis.text(
                x1,
                y1,
                f"false_negative | target={detail.target_category_id}",
                color="white",
                fontsize=7,
                bbox={"facecolor": "magenta", "alpha": 0.8, "pad": 1},
            )
    for detail, prediction in prediction_details_by_image[image_id]:
        color = status_colors.get(detail.status, "gray")
        x, y, width, height = prediction.bbox_xywh
        axis.add_patch(
            Rectangle(
                (x, y),
                width,
                height,
                fill=False,
                edgecolor=color,
                linewidth=2.5,
            )
        )
        axis.text(
            x,
            y,
            f"{detail.status}@{matching_iou_threshold:.2f}\n"
            f"cat={detail.predicted_category_id} | {detail.final_score:.2f}",
            color="white",
            fontsize=7,
            verticalalignment="bottom",
            bbox={"facecolor": color, "alpha": 0.8, "pad": 1},
        )
    axis.set_title(
        f"image={image_id} | correct={status_counts['correct']} | "
        f"FP={status_counts['false_positive']} | "
        f"FN={status_counts['false_negative']} | "
        f"misclassified={status_counts['misclassified']}",
        fontsize=9,
    )

figure.suptitle(
    f"validation prediction review: {len(image_ids_to_plot)} images",
    fontsize=16,
)
figure.tight_layout(rect=(0, 0, 1, 0.99))
grid_name = (
    "selected_validation_grid.png"
    if validation_preview_image_ids
    else "false_positive_images_grid.png"
)
error_grid_path = validation_output_dir / grid_name
figure.savefig(error_grid_path, dpi=150, bbox_inches="tight")
print(f"saved: {error_grid_path}")